
# 📒 Experiment-Auswertung – Notebook in **Kacheln** (Schritt-für-Schritt)

Dieses Notebook führt die Auswertung in **einzelnen Kacheln (Zellen)** aus.
- **Eingabe (Quelle):** `INPUT_ROOT_PATH` (kann **direkt auf den Projekt-Output** zeigen **oder** auf einen Ordner, der **noch eine `Output/`-Ebene** enthält – das Notebook erkennt das automatisch).
- **Ablage (Ergebnisse):** **aktuelles Arbeitsverzeichnis** (`Path.cwd()`) **+** Ordner **`TARGET_ALGORITHM_TARGET_PROFILE`** (z. B. `LSTM_EDGE`).

---
**Ablauf:**
1. Parameter & Ordnerstruktur
2. Hilfsfunktionen laden
3. Daten laden & aufbereiten
4. Visualisieren (interaktive Plotly-Grids als HTML)
5. Ergebnisse/Protokoll speichern
6. Ordnerstruktur anzeigen


In [ ]:

# ============================
# 1) Parameter & Ordnerstruktur
# ============================
from pathlib import Path
import os
from datetime import datetime

# Basis: Das Notebook-Verzeichnis
PROJECT_ROOT = Path.cwd()

# ---- Eingabe-Root (Quelle der Daten) ----
# Beispiel: r"C:\Users\...\MA\Ergebnisse"  (mit oder ohne zusätzlichem \Output am Ende)
# Leer lassen => Standard ist PROJECT_ROOT / "Output"
INPUT_ROOT_PATH = r"C:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Ergebnisse"
INPUT_ROOT_PATH = r"C:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device"

# Aufzulösender Eingabeordner (wo die Quelldateien liegen)
CANDIDATE_ROOT = Path(INPUT_ROOT_PATH) if INPUT_ROOT_PATH else PROJECT_ROOT / "Output"
# Auto-Detection: Falls unter CANDIDATE_ROOT kein Error_Metrics liegt, aber unter CANDIDATE_ROOT/Output schon, nimm die zweite Variante
if not (CANDIDATE_ROOT / "Error_Metrics").exists() and (CANDIDATE_ROOT / "Output" / "Error_Metrics").exists():
    DATA_ROOT = CANDIDATE_ROOT / "Output"
else:
    DATA_ROOT = CANDIDATE_ROOT

# ---- Notebook-Parameter: bitte hier anpassen ----
TARGET_ALGORITHM = "xgboost"     # 'lstm', 'cnn1d', 'random_forest', 'xgboost'
TARGET_PROFILE   = "edge"     # 'edge' oder 'server'
SUMMARY_FILENAME = "Experiment_Summary.csv"

# NEU: nach welchen model_variant(s) gefiltert werden soll.
# Beispiele: "model.joblib", "model.json,model.joblib", "" (leer = kein Filter)
MODEL_VARIANTS = ""  # "" oder "model.keras,model.json,model.joblib

# Falls Pfade in CSVs von Linux zu Windows gemappt werden müssen:
LINUX_BASE_PATH_TO_REPLACE = "/home/pi/ML_Edge_Device/Output"

# Welche Metriken als 3D-Grid geplottet werden sollen (interaktiv als HTML)
METRICS_FOR_PLOTTING = {
    "avg_inference_time_s": "Durchschnittliche Inferenzzeit (s)",
    "avg_total_time_s": "Durchschnittliche Gesamtzeit (s)",
    "avg_cpu_percent": "Durchschnittliche CPU-Auslastung (%)",
    "avg_ram_percent": "Durchschnittliche RAM-Auslastung (%)",
    "model_size_mb": "Modellgröße (MB)"
}

# ---- Ablageordner für Notebook-Ergebnisse ----
REPORT_DIRNAME = f"{TARGET_ALGORITHM}_{TARGET_PROFILE}".upper()
REPORT_DIR     = PROJECT_ROOT / REPORT_DIRNAME
FIG_DIR        = REPORT_DIR / "figs"
EXPORT_DIR     = REPORT_DIR / "exports"
LOG_DIR        = REPORT_DIR / "logs"

for p in [REPORT_DIR, FIG_DIR, EXPORT_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Projekt-Root (Speicherort für neue Dateien):", PROJECT_ROOT)
print("Eingabe-Root (Quelle, auto-detektiert):     ", DATA_ROOT)
print("Report-Ordner (CWD + Name):                 ", REPORT_DIR)
print("Unterordner (figs, exports, logs) wurden angelegt, falls nicht vorhanden.")


Projekt-Root (Speicherort für neue Dateien): c:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device\evaluation
Eingabe-Root (Quelle, auto-detektiert):      C:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device\Output
Report-Ordner (CWD + Name):                  c:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device\evaluation\XGBOOST_EDGE
Unterordner (figs, exports, logs) wurden angelegt, falls nicht vorhanden.


In [12]:
# ==================
# 2) Hilfsfunktionen
# ==================
from pathlib import Path
from typing import Optional
import pandas as pd
import json
import numpy as np
import plotly.graph_objects as go
import os
from datetime import datetime
import io

# --------- Robuste CSV-Ladefunktion (kompatibel mit verschiedenen pandas-Versionen) ---------
def read_csv_robust(path):
    """
    Liest CSV robust ein:
      * bevorzugt: encoding_errors='ignore' + on_bad_lines='skip' (neuere pandas)
      * Fallback: error_bad_lines=False, warn_bad_lines=False (ältere pandas)
      * Letzter Ausweg: Datei manuell mit errors='ignore' lesen und StringIO an pd.read_csv übergeben
    """
    # Versuch 1: moderne Parameter
    try:
        return pd.read_csv(path, engine="python", encoding="utf-8",
                           encoding_errors="ignore", on_bad_lines="skip")
    except TypeError:
        # Versuch 2: ältere Parameter
        try:
            return pd.read_csv(path, engine="python", encoding="utf-8",
                               error_bad_lines=False, warn_bad_lines=False)
        except TypeError:
            # Versuch 3: manuell einlesen und erneut parsen
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                data = f.read()
            return pd.read_csv(io.StringIO(data), engine="python")
    except UnicodeDecodeError:
        # Versuch mit latin1
        try:
            return pd.read_csv(path, engine="python", encoding="latin1", on_bad_lines="skip")
        except TypeError:
            return pd.read_csv(path, engine="python", encoding="latin1",
                               error_bad_lines=False, warn_bad_lines=False)

# Mapping Algorithmus -> Ordnername wie in eurer Output-Struktur
def algorithm_folder_name(algorithm: str) -> str:
    m = {
        "cnn1d": "CNN1D",
        "lstm": "LSTM",
        "random_forest": "Random_Forest",
        "xgboost": "XGBoost",
    }
    return m.get(str(algorithm).lower(), str(algorithm).upper())

def resolve_json_fallback(run_folder: Path) -> Optional[str]:
    cand = list(run_folder.rglob("ErrorMetrics_*.json"))
    return str(cand[0]) if cand else None

def resolve_predictions_fallback(run_folder: Path) -> Optional[str]:
    cand = list(run_folder.rglob("StepPredictions_*.csv"))
    return str(cand[0]) if cand else None

def load_and_process_data(output_data_path: Path,
                          summary_filename: str,
                          algorithm: str,
                          profile: str,
                          linux_base_path: str) -> pd.DataFrame:
    '''
    Liest die zentrale Summary, filtert nach Algorithmus & Profil,
    öffnet pro run_id die Detaildateien, korrigiert Linux->Windows Pfade,
    berechnet Durchschnitts-Metriken und gibt einen DataFrame zurück.
    '''
    summary_path = output_data_path / "Error_Metrics" / summary_filename
    if not summary_path.exists():
        print("FEHLER: Konnte Summary nicht finden:", summary_path)
        return pd.DataFrame()

    print("Verwende Summary:", summary_path)
    df_summary = read_csv_robust(summary_path)
    if df_summary.empty:
        print("WARNUNG: Summary ist leer.")
        return pd.DataFrame()

    filtered_summary = df_summary[(df_summary["algorithm"] == algorithm) & (df_summary["profile"] == profile)]
    if filtered_summary.empty:
        print(f"Keine Experimente für Algorithmus='{algorithm}' und Profil='{profile}' gefunden.")
        return pd.DataFrame()

    results = []
    algo_folder = algorithm_folder_name(algorithm)

    print(f"Datenaufbereitung: {len(filtered_summary)} passende Experimente gefunden. Beginne Verarbeitung...")
    for _, summary_row in filtered_summary.iterrows():
        run_id = str(summary_row.get("run_id", "")).strip()
        if not run_id:
            continue

        run_record = summary_row.to_dict()
        run_record["run_id"] = run_id

        run_folder = output_data_path / algo_folder / run_id
        if not run_folder.is_dir():
            print(f"  - Warnung: Verzeichnis für run_id '{run_id}' nicht gefunden: {run_folder}")
            continue

        # Detail-CSV der Inferenzläufe laden
        run_metrics_csv = run_folder / "Error_Metrics" / "ErrorMetrics_all_runs.csv"
        if not run_metrics_csv.exists():
            print(f"  - Warnung: ErrorMetrics_all_runs.csv fehlt für run_id '{run_id}'")
            continue

        df_run_details = read_csv_robust(run_metrics_csv)
        if df_run_details.empty:
            continue

        run_details_row = df_run_details.iloc[0]

        # Pfadkorrektur Linux->Windows
        predictions_path_linux = str(run_details_row.get("predictions_file_path", ""))
        json_path_linux        = str(run_details_row.get("json_path", ""))

        predictions_path = predictions_path_linux.replace(linux_base_path, str(output_data_path))
        json_path        = json_path_linux.replace(linux_base_path, str(output_data_path))

        # Fallbacks
        if not Path(json_path).exists():
            alt = resolve_json_fallback(run_folder)
            if alt:
                print(f"    JSON-Fallback genutzt für run_id {run_id}: {alt}")
                json_path = alt
        if not Path(predictions_path).exists():
            alt = resolve_predictions_fallback(run_folder)
            if alt:
                print(f"    Predictions-Fallback genutzt für run_id {run_id}: {alt}")
                predictions_path = alt

        try:
            with open(json_path, "r", encoding="utf-8") as f:
                error_data = json.load(f)
            if "metrics" in error_data:
                run_record.update(error_data["metrics"])
        except FileNotFoundError:
            print(f"  - Warnung: JSON nicht gefunden (Pfadmapping prüfen) für run_id '{run_id}'.\n"
                  f"    JSON erwartet: {json_path}")
            continue
        except Exception as e:
            print(f"  - Warnung: JSON-Fehler bei run_id '{run_id}': {type(e).__name__} - {e}")
            continue

        try:
            pred_df = read_csv_robust(predictions_path)
        except FileNotFoundError:
            print(f"  - Warnung: Predictions-CSV nicht gefunden (Pfadmapping prüfen) für run_id '{run_id}'.\n"
                  f"    CSV erwartet: {predictions_path}")
            continue
        except Exception as e:
            print(f"  - Warnung: CSV-Fehler bei run_id '{run_id}': {type(e).__name__} - {e}")
            continue

        # Mittelwerte (erste Zeile optional überspringen)
        pred_df_for_avg = pred_df.iloc[1:].copy() if len(pred_df) > 1 else pred_df.copy()
        avg_cols = ["inference_time_s", "total_time_s", "cpu_percent", "ram_percent", "ram_mb"]
        for col in avg_cols:
            if col in pred_df_for_avg.columns:
                numeric_series = pd.to_numeric(pred_df_for_avg[col], errors="coerce")
                run_record[f"avg_{col}"] = numeric_series.mean()

        # komplette Spalten als Liste beilegen (optional)
        for col in pred_df.columns:
            try:
                run_record[f"{col}_list"] = pred_df[col].tolist()
            except Exception:
                pass

        results.append(run_record)

    if not results:
        print("Datenaufbereitung abgeschlossen, aber keine Runs konnten verarbeitet werden.")
        return pd.DataFrame()

    print(f"Datenaufbereitung erfolgreich: {len(results)} Experiment(e) verarbeitet.")
    return pd.DataFrame(results)


def visualize_data(df: pd.DataFrame, metrics_to_plot: dict, fig_dir: Path):
    '''
    Erstellt für jede gewünschte Metrik ein 3D-Grid (Horizon x Lags) als interaktives HTML.
    Dateien werden in fig_dir gespeichert.
    '''
    fig_dir.mkdir(parents=True, exist_ok=True)
    if df is None or df.empty:
        print("Visualisierung übersprungen (keine Daten).")
        return

    # sicherstellen, dass 'lags' und 'horizon' numerisch sind
    if "lags" in df.columns:
        df["lags"] = pd.to_numeric(df["lags"], errors="coerce")
    if "horizon" in df.columns:
        df["horizon"] = pd.to_numeric(df["horizon"], errors="coerce")

    print(f"Visualisierung: Erstelle bis zu {len(metrics_to_plot)} interaktive Plots...")
    for metric, title in metrics_to_plot.items():
        if metric not in df.columns or not pd.api.types.is_numeric_dtype(df[metric]):
            print(f"  - Hinweis: '{metric}' nicht numerisch/fehlend -> übersprungen.")
            continue

        try:
            pivot_df = df.pivot_table(index="horizon", columns="lags", values=metric, aggfunc="mean")
            fig = go.Figure(data=[
                go.Surface(z=pivot_df.values,
                           x=pivot_df.columns,
                           y=pivot_df.index,
                           colorbar=dict(title=metric.replace("_", " ").title()))
            ])
            fig.update_layout(
                title=title,
                scene=dict(xaxis_title="Lags", yaxis_title="Horizon", zaxis_title=metric),
                width=900, height=750, margin=dict(l=65, r=50, b=65, t=90)
            )

            out_file = fig_dir / f"interactive_grid_plot_{metric}.html"
            fig.write_html(str(out_file))
            print(f"    -> Gespeichert: {out_file.name}")
        except Exception as e:
            print(f"    -> Fehler bei '{metric}': {e}")


def save_dataframe(df: pd.DataFrame, export_dir: Path, filename: str = "processed_runs.csv"):
    export_dir.mkdir(parents=True, exist_ok=True)
    out_csv = export_dir / filename
    df.to_csv(out_csv, index=False, encoding="utf-8")
    print(f"DataFrame exportiert: {out_csv}")
    return out_csv


def dump_report_meta(report_dir: Path,
                     target_algorithm: str,
                     target_profile: str,
                     df: pd.DataFrame | None):
    meta = {
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "report_dir": str(report_dir),
        "target_algorithm": target_algorithm,
        "target_profile": target_profile,
        "num_rows": int(len(df) if df is not None else 0),
        "columns": list(df.columns) if df is not None else [],
    }
    meta_path = report_dir / "report_meta.json"
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)
    print(f"Report-Metadaten gespeichert: {meta_path.name}")
    return meta_path


def write_dir_tree(root: Path, outfile: Path):
    lines = [f"Verzeichnisstruktur für: {root}", "=" * 60]
    for base, dirs, files in os.walk(root):
        base_p = Path(base)
        indent = "  " * (len(base_p.relative_to(root).parts))
        lines.append(f"{indent}{base_p.name}/")
        for d in sorted(dirs):
            lines.append(f"{indent}  {d}/")
        for f in sorted(files):
            lines.append(f"{indent}  {f}")
    outfile.write_text("\n".join(lines), encoding="utf-8")
    print(f"Ordnerstruktur gespeichert: {outfile.name}")
    return outfile


In [13]:

# =============================
# 3) Daten laden & aufbereiten
# =============================
df_processed = load_and_process_data(
    output_data_path=DATA_ROOT,
    summary_filename=SUMMARY_FILENAME,
    algorithm=TARGET_ALGORITHM,
    profile=TARGET_PROFILE,
    linux_base_path=LINUX_BASE_PATH_TO_REPLACE
)

# Vorschau
if df_processed is not None and not df_processed.empty:
    display(df_processed.head(10))
else:
    print("Keine Daten geladen/aufbereitet.")


Verwende Summary: C:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device\Output\Error_Metrics\Experiment_Summary.csv
Datenaufbereitung: 36 passende Experimente gefunden. Beginne Verarbeitung...
Datenaufbereitung erfolgreich: 36 Experiment(e) verarbeitet.


,algorithm,profile,lags,horizon,model_variant,avg_inference_time_ms,avg_total_time_ms,avg_cpu_percent,avg_ram_percent,model_size_mb,...,pred_h7_list,pred_h8_list,pred_h9_list,pred_h10_list,pred_h11_list,pred_h12_list,pred_h13_list,pred_h14_list,pred_h15_list,pred_h16_list
0,xgboost,edge,1,1,model.json,2.226818,33.142860,27.601558,71.272414,0.3301,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,xgboost,edge,1,4,model.joblib,6.873701,37.379358,32.379328,57.789655,0.2175,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,xgboost,edge,1,7,model.joblib,11.255774,42.614420,36.231321,53.900000,0.3842,...,"[0.296708345413208, 0.296708345413208, 0.29670...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,xgboost,edge,1,10,model.joblib,14.878750,45.211738,39.296361,52.800000,0.5521,...,"[0.2868290245532989, 0.2868290245532989, 0.286...","[-0.0261194314807653, -0.0261194314807653, -0....","[1.413179636001587, 1.413179636001587, 1.41317...","[0.2421101778745651, 0.2421101778745651, 0.242...",NaN,NaN,NaN,NaN,NaN,NaN
4,xgboost,edge,1,13,model.joblib,18.477305,49.219110,41.928260,52.800000,0.7107,...,"[0.2949931919574737, 0.2949931919574737, 0.294...","[-0.0297516398131847, -0.0297516398131847, -0....","[1.3770759105682373, 1.3770759105682373, 1.377...","[0.2330873608589172, 0.2330873608589172, 0.233...","[0.0488335229456424, 0.0488335229456424, 0.048...","[0.1652346700429916, 0.1652346700429916, 0.165...","[1.207082986831665, 1.207082986831665, 1.20708...",NaN,NaN,NaN
5,xgboost,edge,1,16,model.joblib,22.457480,53.396034,44.483182,51.577586,0.8737,...,"[0.2881912589073181, 0.2881912589073181, 0.288...","[-0.0258118044584989, -0.0258118044584989, -0....","[1.3531935214996338, 1.3531935214996338, 1.353...","[0.247044563293457, 0.247044563293457, 0.24704...","[0.0550010725855827, 0.0550010725855827, 0.055...","[0.1709193736314773, 0.1709193736314773, 0.170...","[1.2120739221572876, 1.2120739221572876, 1.212...","[0.0054236771538853, 0.0054236771538853, 0.005...","[-0.0061219851486384, -0.0061219851486384, -0....","[0.427316814661026, 0.427316814661026, 0.42731..."
6,xgboost,edge,4,1,model.json,2.362782,41.937354,27.689781,50.000000,0.3425,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,xgboost,edge,4,4,model.joblib,6.397432,45.692147,30.626471,50.474545,0.2374,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,xgboost,edge,4,7,model.joblib,10.837361,50.488224,33.668827,51.300000,0.4268,...,"[0.4649277031421661, 0.4649277031421661, 0.464...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,xgboost,edge,4,10,model.joblib,15.049311,53.563065,36.903950,51.500000,0.6158,...,"[0.4651572406291961, 0.4651572406291961, 0.465...","[0.1848817318677902, 0.1848817318677902, 0.184...","[0.3670307099819183, 0.3670307099819183, 0.367...","[0.3735662698745727, 0.3735662698745727, 0.373...",NaN,NaN,NaN,NaN,NaN,NaN


In [14]:

# =========================
# 4) Visualisierung (Grids)
# =========================
visualize_data(df_processed, METRICS_FOR_PLOTTING, FIG_DIR)


Visualisierung: Erstelle bis zu 5 interaktive Plots...
    -> Gespeichert: interactive_grid_plot_avg_inference_time_s.html
    -> Gespeichert: interactive_grid_plot_avg_total_time_s.html
    -> Gespeichert: interactive_grid_plot_avg_cpu_percent.html
    -> Gespeichert: interactive_grid_plot_avg_ram_percent.html
    -> Gespeichert: interactive_grid_plot_model_size_mb.html


In [15]:

# ===============================
# 5) Exporte & Protokoll speichern
# ===============================
if df_processed is not None and not df_processed.empty:
    csv_path = save_dataframe(df_processed, EXPORT_DIR, filename="processed_runs.csv")
else:
    csv_path = None

meta_path = dump_report_meta(REPORT_DIR, TARGET_ALGORITHM, TARGET_PROFILE, df_processed)
print("Fertig.")


DataFrame exportiert: c:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device\evaluation\XGBOOST_EDGE\exports\processed_runs.csv
Report-Metadaten gespeichert: report_meta.json
Fertig.


In [16]:

# ========================
# 6) Ordnerstruktur anzeigen
# ========================
_ = write_dir_tree(REPORT_DIR, REPORT_DIR / "directory_tree.txt")
print(f"Ergebnisordner: {REPORT_DIR}")


Ordnerstruktur gespeichert: directory_tree.txt
Ergebnisordner: c:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device\evaluation\XGBOOST_EDGE
